# Optimizing LLM-Powered Applications: Cost, Tokens & Performance

**A hands-on workshop.** You'll take a small, real AI *coding agent* and make
it cheaper and faster — then **prove** the improvement with real numbers, not
guesses.

Building an LLM app that *works* is the easy part. Making it **cost-effective,
fast, and production-ready** is the real job. In this notebook you will:

1. Clone a lightweight coding agent (built from scratch — no heavy frameworks).
2. Run it on a fixed set of tasks and record **tokens, cost, and speed**.
3. Turn on optimizations **one at a time** and watch the numbers drop.
4. **Stack** optimizations and measure the combined win.

Everything runs in the browser — no local Python or VS Code install needed.
The only thing you provide is an **OpenRouter API key** (free to create).

> Every number in this notebook is a **real** token count from the model's API
> response — never estimated. That's the whole point: optimizations are judged
> by evidence.

## How this notebook works

The agent is normally a terminal chat app (`uv run coding-agent`). Here we drive
the **exact same code** in-process, one prompt at a time, so we can capture the
precise tokens and cost of each run. Whenever you see a cell run the agent, the
terminal equivalent is shown in the notes.

**The plan:** run the same fixed prompts as a *baseline*, then re-run them with
each optimization enabled and compare. Because the task never changes, any drop
in tokens/cost is caused by the optimization — nothing else.

Run the cells **top to bottom.** Start with setup.

---
## Step 1 — Clone the agent

Downloads the public repository. Safe to re-run (it reuses an existing clone).

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shrijayan/coding-agent.git"
REPO_BRANCH = "main"

BASE_DIR = os.getcwd()                       # /content on Colab
REPO_DIR = os.path.join(BASE_DIR, "coding-agent")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning", REPO_URL, "...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print("Repo already present at", REPO_DIR)

# Make `import coding_agent` work without a full package install.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("REPO_DIR =", REPO_DIR)

## Step 2 — Install dependencies

A handful of small libraries. Takes a few seconds.

In [ ]:
import subprocess, sys

DEPS = ["anthropic>=0.120.0", "openai>=2.48.0", "python-dotenv>=1.2.2", "pyyaml>=6.0.3"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)

# For the comparison tables and charts (usually already present in Colab).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=False)

print("Dependencies installed.")

## Step 3 — Add your OpenRouter API key

Get a key at **https://openrouter.ai/keys**.

**Recommended (Colab):** click the key icon in the left sidebar, add a secret
named `OPENROUTER_API_KEY`, and enable notebook access. Then run the cell — it
picks the key up automatically.

**Otherwise:** the cell prompts you to paste the key into a hidden input box
(it is never printed or stored in the notebook).

In [ ]:
import os

def _load_openrouter_key():
    # 1) Colab Secrets (preferred)
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get("OPENROUTER_API_KEY")
        if val:
            return val.strip(), "Colab Secrets"
    except Exception:
        pass
    # 2) Already exported in the environment
    if os.environ.get("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"].strip(), "environment"
    # 3) Hidden manual prompt
    import getpass
    return getpass.getpass("Paste your OpenRouter API key (hidden): ").strip(), "manual input"

_key, _src = _load_openrouter_key()
assert _key, "No API key provided — re-run this cell and paste your key."
os.environ["OPENROUTER_API_KEY"] = _key
print(f"OpenRouter key loaded from {_src} (length {len(_key)}).")

## Step 4 — Configure the agent

We pin the provider to OpenRouter and set the agent's required settings. These
mirror the project's `.env` file. You can tweak them, but the defaults are fine.

In [ ]:
import os

os.environ["AGENT_PROVIDER"] = "openrouter"
os.environ.setdefault("AGENT_MAX_ITERATIONS", "25")
os.environ.setdefault("AGENT_BASH_TIMEOUT_SECONDS", "60")
# Conversation-summary thresholds (keep_recent MUST be < threshold).
os.environ.setdefault("AGENT_SUMMARY_THRESHOLD_MESSAGES", "8")
os.environ.setdefault("AGENT_SUMMARY_KEEP_RECENT_MESSAGES", "4")

print("Agent configured (provider = openrouter).")

## Step 5 — Sanity check & connectivity

Loads the agent's config and sends a tiny 1-word request to the model. If your
key is wrong, or a model slug in `models.yaml` isn't a real OpenRouter model,
this fails **here** — loudly — instead of halfway through a demo.

In [ ]:
from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.models_config import read_models_yaml
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS

cfg = Config.from_env()
pricing = PricingTable.load()
pricing.require(cfg.model)

print("Base model      :", f"{cfg.provider} / {cfg.model}")
print("Session cost cap :", cfg.session_cost_cap_usd, "USD")
print("Optimizations available:", ", ".join(AVAILABLE_OPTIMIZATIONS) or "none")

# Live ping through the agent's own client.
from coding_agent.llm.factory import build_llm_client
from coding_agent.llm.messages import Message, TextPart

_client = build_llm_client(cfg)
try:
    _r = _client.send(
        system="Reply with the single word: ok",
        messages=[Message(role="user", parts=[TextPart("ping")])],
        tools=[],
    )
    print("\nConnectivity OK — model replied:", repr(_r.text.strip()[:60]))
    print("Reported usage :", _r.usage)
except Exception as e:
    print("\nConnectivity FAILED:", type(e).__name__, "-", str(e)[:300])
    print("Fix your key, or edit coding-agent/src/coding_agent/models.yaml so the")
    print("model slugs are real OpenRouter models (https://openrouter.ai/models).")

---
## The measurement harness

This cell defines two small helpers we use for the rest of the notebook:

- **`WorkshopSession`** — one agent session with a chosen set of optimizations.
  `.ask(prompt)` runs a turn, `.usage_report()` prints the same output as the
  `/usage` command in the real terminal app.
- **`reset_playground()`** — a clean scratch folder the agent may create and
  edit files in (its tools act on the current directory).

You don't need to read the code to do the workshop — but it's short and it's
exactly how the project's own benchmark runner drives the agent.

In [ ]:
import os, shutil, time
from pathlib import Path

from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.metrics.usage import UsageTracker
from coding_agent.agent.factory import build_agent
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
from coding_agent.optimizations.registry import OptimizationRegistry
from coding_agent.commands.usage_command import UsageCommand
from coding_agent.models_config import load_catalog_metadata, read_models_yaml

PLAYGROUND = Path(BASE_DIR) / "playground"

def reset_playground():
    """Wipe and recreate the agent's scratch directory, then move into it."""
    if PLAYGROUND.exists():
        shutil.rmtree(PLAYGROUND)
    PLAYGROUND.mkdir(parents=True, exist_ok=True)
    os.chdir(PLAYGROUND)

class WorkshopSession:
    """An agent session with a chosen set of optimizations enabled."""

    def __init__(self, optimizations=None, enforce_cost_cap=False):
        self.enabled = list(optimizations or [])
        self.config = Config.from_env()
        self.pricing = PricingTable.load()
        self.pricing.require(self.config.model)
        self.usage = UsageTracker()

        bundle = OptimizationRegistry(AVAILABLE_OPTIMIZATIONS).resolve(self.enabled)
        self.agent = build_agent(
            self.config, self.usage, bundle,
            pricing=self.pricing if enforce_cost_cap else None,
        )

        self.configured_models = None
        self.routing_tracker = None
        if "hybrid-routing" in self.enabled:
            from coding_agent.optimizations import hybrid_routing
            from coding_agent.optimizations.routing.tiers import load_tiers
            tiers = load_tiers()
            for t in tiers:
                self.pricing.require(t.model or self.config.model)
            self.configured_models = [t.model or self.config.model for t in tiers]
            self.routing_tracker = hybrid_routing.get_tracker()
            for w in hybrid_routing.get_warnings():
                print("warning>", w)

        self._meta = load_catalog_metadata(read_models_yaml())

    def ask(self, prompt, show_tools=True):
        """Run one turn: print the tool calls and the final answer."""
        PLAYGROUND.mkdir(parents=True, exist_ok=True)
        os.chdir(PLAYGROUND)
        start = time.perf_counter()

        def on_tool(name, tool_input):
            if show_tools:
                print(f"  [tool] {name}({tool_input})")

        print(f"you> {prompt}")
        answer = self.agent.run_turn(prompt, on_tool_call=on_tool)
        ms = (time.perf_counter() - start) * 1000
        print(f"\nagent> {answer}")
        print(f"  \u21b3 {self.config.model} \u00b7 {self.usage.llm_calls} LLM calls "
              f"(session) \u00b7 {self.usage.total.total_tokens:,} tokens (session) "
              f"\u00b7 {ms:.0f}ms this turn\n")
        return answer

    def usage_report(self):
        """Print the same thing the /usage terminal command prints."""
        cmd = UsageCommand(
            tracker=self.usage, pricing=self.pricing, config=self.config,
            enabled_optimizations=self.enabled,
            configured_models=self.configured_models,
            model_metadata=self._meta,
            cost_cap_usd=self.config.session_cost_cap_usd,
        )
        print(cmd.run())

    def routing_report(self):
        """Print the /metrics command output (only meaningful with routing)."""
        if not self.routing_tracker:
            print("Routing is not enabled for this session.")
            return
        from coding_agent.commands.metrics_command import RoutingMetricsCommand
        print(RoutingMetricsCommand(tracker=self.routing_tracker, pricing=self.pricing).run())

    def metrics(self, label=None):
        """The measured numbers for this session, as a plain dict."""
        total = self.usage.total
        cost = sum(self.pricing.cost_for(u, m) for m, u in self.usage.by_model.items())
        return {
            "scenario": label or (", ".join(self.enabled) or "base"),
            "optimizations": ", ".join(self.enabled) or "none",
            "llm_calls": self.usage.llm_calls,
            "tool_calls": self.usage.tool_calls,
            "input_tokens": total.input_tokens,
            "output_tokens": total.output_tokens,
            "total_tokens": total.total_tokens,
            "cost_usd": round(cost, 6),
        }

print("Harness ready: WorkshopSession, reset_playground().")

In [ ]:
def run_scenario(label, optimizations, prompts, show_tools=False):
    """Run the SAME prompts through a fresh agent with the chosen optimizations."""
    reset_playground()
    session = WorkshopSession(optimizations=optimizations)
    opt_label = ", ".join(optimizations) or "none"
    print(f"=== Scenario: {label}  (optimizations: {opt_label}) ===")
    for p in prompts:
        session.ask(p, show_tools=show_tools)
    m = session.metrics(label)
    print(f"--- {label}: {m['total_tokens']:,} tokens \u00b7 ${m['cost_usd']:.4f} "
          f"\u00b7 {m['llm_calls']} LLM calls ---\n")
    return {"session": session, "metrics": m}

def _pct(base, new):
    return 0.0 if base == 0 else (base - new) / base * 100.0

def compare(baseline, *others):
    """Show baseline vs one or more optimized runs, with % saved."""
    rows = [baseline["metrics"]] + [o["metrics"] for o in others]
    base = baseline["metrics"]
    try:
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(rows)
        df["tokens_saved_%"] = df["total_tokens"].apply(
            lambda t: round(_pct(base["total_tokens"], t), 1))
        df["cost_saved_%"] = df["cost_usd"].apply(
            lambda c: round(_pct(base["cost_usd"], c), 1))
        cols = ["scenario", "llm_calls", "tool_calls", "total_tokens",
                "tokens_saved_%", "cost_usd", "cost_saved_%"]
        display(df[cols])
        return df
    except Exception:
        print(f"{'scenario':<26}{'tokens':>10}{'saved%':>9}{'cost$':>11}{'saved%':>9}")
        for r in rows:
            print(f"{r['scenario']:<26}{r['total_tokens']:>10,}"
                  f"{_pct(base['total_tokens'], r['total_tokens']):>8.1f}%"
                  f"{r['cost_usd']:>11.4f}"
                  f"{_pct(base['cost_usd'], r['cost_usd']):>8.1f}%")

print("Ready: run_scenario(...), compare(...).")

---
## Meet the base agent

No optimizations yet. Watch the `[tool]` lines — the agent really reads and
writes files and runs commands; it doesn't just *describe* what to do. Edit the
prompt and re-run to experiment.

In [ ]:
reset_playground()
base_demo = WorkshopSession()   # no optimizations
base_demo.ask("Create hello.py that prints 'Hello, workshop!', then show me its contents.")

### The `/usage` command

In the terminal app you'd type `/usage` to see tokens, cost, and which
optimizations are on. Here's the exact same report for the session above.

In [ ]:
base_demo.usage_report()

---
## The shared prompt set

To compare fairly, every configuration runs the **same fixed conversation**. It
builds a small module across several turns — each turn depends on the previous
ones. That growing history is exactly what *conversation summarization*
compresses, and the routine edits are what *model routing* sends to a cheaper
model.

> The task never changes, only the optimization does — so any difference in the
> numbers is caused by the optimization, not by a different prompt.

In [ ]:
DEMO_PROMPTS = [
    "Create a file calculator.py with a function add(a, b) that returns their sum. Keep it minimal.",
    "Add a subtract(a, b) function to calculator.py.",
    "Add multiply(a, b) and divide(a, b) to calculator.py. divide must raise ValueError on division by zero.",
    "Create test_calculator.py with pytest tests for all four functions, including the divide-by-zero case.",
    "List the files you created and give a one-line summary of what calculator.py now contains.",
]
print(f"{len(DEMO_PROMPTS)} prompts ready.")

---
## Baseline — the un-optimized agent

Run the shared prompts with **no** optimizations and record the numbers. Every
later scenario is measured against this. (This calls the model several times, so
give it a moment.)

In [ ]:
baseline = run_scenario("base agent", [], DEMO_PROMPTS)
baseline["session"].usage_report()

---
## Optimization 1 — Conversation summarization

Every turn, the agent resends the **entire** conversation history to the model.
As the chat grows, so do the input tokens on *every* call — you pay again and
again to resend old context.

**Conversation summarization** compresses older messages into a short running
summary once history passes a threshold, keeping only the last few messages
verbatim. Fewer input tokens per call → lower cost, with recent context intact.

*Terminal equivalent:* `uv run coding-agent --enable conversation-summary`

In [ ]:
opt_summary = run_scenario("+ conversation-summary", ["conversation-summary"], DEMO_PROMPTS)
compare(baseline, opt_summary)

---
## Optimization 2 — Model routing (hybrid routing)

Not every request needs your most capable (most expensive) model. **Hybrid
routing** scores each request's difficulty (free, local) and sends easy/routine
work to a **cheaper** model, escalating to a stronger tier only when a quality
gate flags the cheap answer.

The base agent pays the mid-tier price for *everything*; routing pays cheap-tier
prices for the routine edits. The ladder is defined in `models.yaml`
(`routing.tiers`) — a data edit, no code change.

*Terminal equivalent:* `uv run coding-agent --enable hybrid-routing`

In [ ]:
opt_routing = run_scenario("+ hybrid-routing", ["hybrid-routing"], DEMO_PROMPTS)
compare(baseline, opt_routing)

### The `/metrics` command

With routing on, `/metrics` breaks down which tier answered, how often the
cheap model was enough, and how often it escalated.

In [ ]:
opt_routing["session"].routing_report()

---
## Stack them — combined optimizations

Optimizations **compose**. `conversation-summary` controls *what history is
sent*; `hybrid-routing` controls *which model answers* — different hooks, no
conflict, so both apply at once. This is usually where the biggest savings show
up.

*Terminal equivalent:* `uv run coding-agent --enable conversation-summary,hybrid-routing`

In [ ]:
opt_both = run_scenario("+ both", ["conversation-summary", "hybrid-routing"], DEMO_PROMPTS)
compare(baseline, opt_summary, opt_routing, opt_both)

---
## Scoreboard

The whole story in two charts: total tokens and estimated cost, baseline vs each
optimization vs both.

In [ ]:
runs = [baseline, opt_summary, opt_routing, opt_both]
try:
    import matplotlib.pyplot as plt
    labels = [r["metrics"]["scenario"] for r in runs]
    tokens = [r["metrics"]["total_tokens"] for r in runs]
    costs = [r["metrics"]["cost_usd"] for r in runs]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.bar(labels, tokens); ax1.set_title("Total tokens (lower is better)")
    ax1.tick_params(axis="x", rotation=30)
    ax2.bar(labels, costs); ax2.set_title("Estimated cost, USD (lower is better)")
    ax2.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Chart skipped:", e)
    for r in runs:
        m = r["metrics"]
        print(f"{m['scenario']:<26} {m['total_tokens']:>8,} tokens  ${m['cost_usd']:.4f}")

---
## Coming soon — more optimizations

The team is adding more optimizations to the **same** plug-in system. Each slots
into this notebook with the exact same `run_scenario([...], DEMO_PROMPTS)` call —
no harness changes needed.

- **Prompt optimization & prompt caching** — trim and stabilize the system
  prompt so the provider can cache the static prefix, cutting repeated
  input-token cost on every call.
- **Context-window optimization** — send only the most relevant slice of the
  history/files instead of everything.
- **Agent-loop prevention** — detect repeated or looping tool calls early and
  stop, so a confused model can't burn iterations (and tokens).

When one ships, it shows up in the list below automatically. Add its name to a
`run_scenario` cell and re-run `compare(...)` to measure it.

In [ ]:
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
print("Optimizations available right now:")
for name in AVAILABLE_OPTIMIZATIONS:
    print("  \u2022", name)

---
## Your turn — free play

Enable any combination of the available optimizations and give the agent your
own task. Try the same prompt with and without an optimization and compare
`usage_report()` output.

In [ ]:
reset_playground()
my_session = WorkshopSession(optimizations=[])   # e.g. ["conversation-summary", "hybrid-routing"]
my_session.ask("Write a Python function that reverses a string without slicing, and show it to me.")
my_session.usage_report()

---
## Appendix — running the real terminal agent

Everything above drives the **same** agent you'd run in a terminal — just
in-process so we can measure it cleanly. To run the actual REPL locally
(outside Colab):

```bash
git clone https://github.com/shrijayan/coding-agent.git
cd coding-agent
uv sync
cp .env.example .env          # add your OPENROUTER_API_KEY
uv run coding-agent --enable conversation-summary,hybrid-routing
```

Inside the REPL, type `/usage` (and `/metrics` with routing enabled) to see the
same numbers this notebook computes. To measure correctness as well as cost,
`uv run coding-agent --benchmark --enable <name>` runs a fixed suite of real
coding tasks.